In [1]:
import pprint
import multiprocessing

NUM_PROCESSORS=multiprocessing.cpu_count()
print("Cpu count: ",NUM_PROCESSORS)

Cpu count:  12


In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
import os.path as osp
import matplotlib.pyplot as plt
#from tqdm.notebook import tqdm
from tqdm import tqdm
import numpy as np
import networkx as nx
from scipy import sparse, stats
from numpy import inf
from scipy.sparse import csgraph

In [4]:
from scipy.sparse import identity
from scipy.sparse import csgraph
from scipy import linalg
from scipy.sparse import csr_matrix
#from scipy.sparse import linalg

In [5]:
import ipynb.fs.full.utils.BarbellGraph as BGraph

In [1]:
DRAW=False
if DRAW:
    G, pos = BGraph.generate_barbell(10,10)
    BGraph.draw_graph(G,pos)

### True Effective Resistance Sparsification

In [2]:
def ER(u,v, L_inv,N):
    x_u = np.zeros((N,))
    x_v = np.zeros((N,)) 
    x_u[u] = 1
    x_v[v] = 1
    d_uv=x_u-x_v
    R_uv=d_uv.dot(L_inv.dot(d_uv)) ## (x_u-x_v)^T*L'*(x_u-x_v)
    return R_uv

def compute_ER(Adj, L_inv):
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(Adj))
    n = np.shape(Adj)[0]
    Reff = sparse.lil_matrix((n,n))
    for orig, end in zip(start_nodes, end_nodes):
        Reff[orig,end] = ER(orig, end, L_inv, n)
    return Reff

def EffectiveResistance(G):
    Adj = nx.adjacency_matrix(G)
    L, D  = csgraph.laplacian(Adj, normed=False, return_diag=True)
    #print(np.allclose(L.todense(), np.diag(D)-Adj)) #verify L=D-A
    L_inv = linalg.pinv(L.todense())
    Reff=compute_ER(Adj, L_inv)
    return Reff
    
#Reff = EffectiveResistance(G)

In [1]:
def TrueERSparsify(G, epsilon=0.5):
    
    print("Computing edge resistances:... ")
    resistance_distances = EffectiveResistance(G)
    print("Finished computing resistances:")
    
    Adj = nx.adjacency_matrix(G)
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(Adj))

    N=Adj.shape[0]
    
    # Calculate the new weights.
    weights = np.maximum(0, weights)
    Re = np.maximum(0, resistance_distances[start_nodes, end_nodes].toarray())
    Pe = weights * Re
    Pe = Pe / np.sum(Pe)    
    Pe = np.squeeze(Pe)
    
    # Rudelson, 1996 Random Vectors in the Isotropic Position
    # (too hard to figure out actual C0)
    C0 = 1 / 30.
    # Rudelson and Vershynin, 2007, Thm. 3.1
    C = 4 * C0
    q = round(N * np.log(N) * 9 * C ** 2 / (epsilon ** 2))
    
#     print(len(start_nodes))
#     print(C0,C,q)

    #        results = stats.rv_discrete(values=(np.arange(np.shape(Pe)[0]), Pe)).rvs(size=int(q))
    results = np.random.choice(np.arange(np.shape(Pe)[0]), int(q), p=list(Pe))
    #spin_counts = stats.itemfreq(results).astype(int)
    spin_counts = np.unique(results, return_counts=True)
    
    #print(results, spin_counts)
    
    per_spin_weights = weights / (q * Pe)
    per_spin_weights[per_spin_weights == inf] = 0
    
    #print(per_spin_weights)

    counts = np.zeros(np.shape(weights)[0])
    #counts[spin_counts[:, 0]] = spin_counts[:, 1]    
    counts[spin_counts[0]] = spin_counts[1]
    
    #print(counts)
    
    new_weights = counts * per_spin_weights

    sparserW = sparse.csc_matrix((np.squeeze(new_weights), (start_nodes, end_nodes)),shape=(N, N))    
    sparserW = sparserW + sparserW.T ##making symmetric
    
    #print(N,q)

    return sparserW, np.count_nonzero(new_weights)
    
# sparserW, nz = TrueERSparsify(G,0.5)
# print(nz) ##directed edges
# Gsparse = nx.from_scipy_sparse_matrix(sparserW)
# BGraph.draw_graph(Gsparse,pos)

### Local Effective Resistance Sparsification

In [64]:
import math
import csrgraph as cg
from random import choice

In [65]:
# G = cg.csrgraph(G, threads=12) 
# node_names = G.names
# walks = G.random_walks(walklen=10, # length of the walks
#                 epochs=100, # how many times to start a walk from each node
#                 start_nodes=0, # the starting node. It is either a list (e.g., [2,3]) or None. If None it does it on all nodes and returns epochs*G.number_of_nodes() walks
#                 return_weight=1.,
#                 neighbor_weight=1.)

In [1]:
# At first initial vertex, v = starting vertex. Then I got the whole graph and edge by adj.csr() where in the rowptr. With the help of 
# I could find how many neighbors are there for node v. Then I could find the neighbors from col. 
def random_walk(adj, s, l):
    v = s;
    rowptr_, col_, _ = adj.csr()
    row_start = rowptr_[v]
    row_end = rowptr_[v + 1]
    for i in range(l):  
        neighbors = col_[row_start:row_end]
        if (neighbors.numel() == 0):
            continue;
        v = choice(list(neighbors))
    return v


# In this part, I only change in in two parts. At first the model parameters and the second is to find a node's degree from adjacency matrix. 
# I calculated the degree of a particular node from the rowptr.  In the rowptr each of the index described how many consecutive vertex are 
# in the col are their neighbors. So degree_node_v = rowptr[v+1] - rowptr[v].  Suppose rowptr = [0, 2, 6, 8]. So it means that
# vertex 0 has neighbors from col[0:2]. vertex 1 has neighbors col[2:6]. So 
def effective_resistance_akp(adj,s, t, eps=0.1, lmbda=0.1):
    l = math.ceil(math.log(4 / (eps * (1 - lmbda))) / math.log(1.0 / lmbda) / 2)
    r = int(math.ceil(40 * l * l * math.log(80 * l) / (eps * eps)))
    delta = 0
    rowptr_, _, _ = adj.csr()
 

    for i in range(l):
        Xis = 0; Xit = 0; Yis = 0; Yit = 0;
        
        for j in range(r):
            v = random_walk(adj, s, i)
            if (v == s):
                Xis+=1
            if (v == t):
                Xit+=1    
        
        for j in range(r):
            v = random_walk(adj, t, i);
            if (v == s):
                Yis+=1;
            if (v == t):
                Yit+=1;
        degree_s = rowptr_[s+1] - rowptr_[s]
        degree_t = rowptr_[t+1] - rowptr_[t]
        #deltai = float(Xis) / G.degree[s] - float(Xit) / G.degree[t] - float(Yis) / G.degree[s] + float(Yit) / G.degree[t]
        deltai = float(Xis) / degree_s - float(Xit) / degree_t - float(Yis) / degree_s + float(Yit) / degree_t
        deltai /= r;
        delta += deltai;

    return max(0,delta)

from tqdm import tqdm

def get_sparse_adj_matrix(adj_t):
    rowptr, col, _ = adj_t.csr()
    Adj = np.zeros((len(rowptr) - 1, len(rowptr) - 1))
    j = 0
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            weight = 1
            Adj[source_vertex, target_vertex] = weight
            j += 1  
    sparse_adj = csr_matrix(Adj)
    return sparse_adj 

def LocalEffectiveResistance(adj_t=None, method=None, eps=0.5, lmbda=0.1):
    Adj = get_sparse_adj_matrix(adj_t)
    if Adj==None:
        start_nodes, end_nodes, weights = sparse.find(Adj)
    else:
        start_nodes, end_nodes, weights = sparse.find(Adj)
    
    nE=len(start_nodes)
    pbar = tqdm(total=nE)
    pbar.set_description(f'Edges')
    Re=np.zeros(nE)
    last_i=-1
    for i in range(nE):
        
        Re[i]=effective_resistance_akp(adj_t, start_nodes[i], end_nodes[i], eps, lmbda)
        
        if (i%1000==0 and i>0) or i==(nE-1):
            pbar.update(i-last_i)
            last_i=i
        
    pbar.close()

    return Re
    
#LocalEffectiveResistance(G)
# print(effective_resistance_akp(G, 21, 20, eps=0.3, lmbda=0.25))

In [1]:
from tqdm import tqdm

def get_sparse_adj_matrix(adj_t):
    rowptr, col, _ = adj_t.csr()
    Adj = np.zeros((len(rowptr) - 1, len(rowptr) - 1))
    j = 0
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            weight = 1
            Adj[source_vertex, target_vertex] = weight
            j += 1  
    sparse_adj = csr_matrix(Adj)
    return sparse_adj 

def LocalEffectiveResistance(adj_t=None, method=None, eps=0.5, lmbda=0.1):
    Adj = get_sparse_adj_matrix(adj_t)
    if Adj==None:
        start_nodes, end_nodes, weights = sparse.find(Adj)
    else:
        start_nodes, end_nodes, weights = sparse.find(Adj)
    
    nE=len(start_nodes)
    pbar = tqdm(total=nE)
    pbar.set_description(f'Edges')
    Re=np.zeros(nE)
    last_i=-1
    for i in range(nE):
        
        Re[i]=effective_resistance_akp(adj_t, start_nodes[i], end_nodes[i], eps, lmbda)
        
        if (i%1000==0 and i>0) or i==(nE-1):
            pbar.update(i-last_i)
            last_i=i
        
    pbar.close()

    return Re
    
#LocalEffectiveResistance(G)

In [68]:
from LocalSpectralParallel import effective_resistance_akp_parallel
from joblib import Parallel, delayed

In [69]:
def LocalEffectiveResistanceParallell(G, Adj=None, method=None, window=50):
    
    if Adj==None:
        Adj = nx.adjacency_matrix(G)
        start_nodes, end_nodes, weights = sparse.find(sparse.tril(Adj))
    else:
        start_nodes, end_nodes, weights = sparse.find(sparse.tril(Adj))
    
    nE=len(start_nodes)
    
    pbar = tqdm(total=nE)
    pbar.set_description(f'Edges')
    
    Re=np.zeros(nE)
    last_i=-1
    
    n_jobs=NUM_PROCESSORS
    window=max(window, n_jobs,NUM_PROCESSORS)
    print("Parallel window: ",window, " Jobs: ",n_jobs)
    
    params={}
    parallel=Parallel(n_jobs=n_jobs, require='sharedmem', prefer="threads")
    for i in range(nE):
            
        params[i]=(G, start_nodes[i], end_nodes[i], 0.9, 0.1) #g, s, t, eps, lambda
        
        if (i%window==0 and i>0) or i==(nE-1):
                        
            results = parallel(
                delayed(effective_resistance_akp_parallel)(key, value[0],value[1],value[2],value[3], value[4]) for key,value in params.items()
            )
            
            results_dict = dict(results)
        
            for key,value in results_dict.items():
                Re[key]=value
            
            pbar.update(i-last_i)
            last_i=i
            
            params={}
    pbar.close()
    return Re
    
#LocalEffectiveResistanceParallell(G)

In [1]:
def LocalERSparsify(Adj,epsilon=0.5, computation='parallel', window=100):
        
    
    start_nodes, end_nodes, weights = sparse.find(Adj)
    
    print("ER computation: ",computation)
    
    if computation=='parallel':
        Re=LocalEffectiveResistanceParallell(G, Adj, window=window)
    else:
        Re=LocalEffectiveResistance(G,Adj)
    
    N=Adj.shape[0]
    
    # Calculate the new weights.
    weights = np.maximum(0, weights)
    Pe = weights * Re
    Pe = Pe / np.sum(Pe)    
    Pe = np.squeeze(Pe)
    
    # Rudelson, 1996 Random Vectors in the Isotropic Position
    # (too hard to figure out actual C0)
    C0 = 1 / 30.
    # Rudelson and Vershynin, 2007, Thm. 3.1
    C = 4 * C0
    q = round(N * np.log(N) * 9 * C ** 2 / (epsilon ** 2))
    
#     print(len(start_nodes))
#     print(C0,C,q)

    #        results = stats.rv_discrete(values=(np.arange(np.shape(Pe)[0]), Pe)).rvs(size=int(q))
    results = np.random.choice(np.arange(np.shape(Pe)[0]), int(q), p=list(Pe))
    #spin_counts = stats.itemfreq(results).astype(int)
    spin_counts = np.unique(results, return_counts=True)
    
    #print(results, spin_counts)
    
    per_spin_weights = weights / (q * Pe)
    per_spin_weights[per_spin_weights == inf] = 0
    
    #print(per_spin_weights)

    counts = np.zeros(np.shape(weights)[0])
    #counts[spin_counts[:, 0]] = spin_counts[:, 1]    
    counts[spin_counts[0]] = spin_counts[1]
    
    #print(counts)
    
    new_weights = counts * per_spin_weights

    sparserW = sparse.csc_matrix((np.squeeze(new_weights), (start_nodes, end_nodes)),shape=(N, N))    
    sparserW = sparserW + sparserW.T ##making symmetric
    
    #print(N,q)

    return sparserW, np.count_nonzero(new_weights)
    

# #sparserW, nz = LocalERSparsify(G,0.5, computation='parallel', window=NUM_PROCESSORS*100)
# sparserW, nz = LocalERSparsify(G,0.5, computation='serial', window=NUM_PROCESSORS*100)
# print(nz) ##directed edges
# Gsparse = nx.from_scipy_sparse_matrix(sparserW)
# BGraph.draw_graph(Gsparse,pos)

# Efficient ER computation batchwise

In [71]:
def local_effective_resistance_akp(G, s, t, eps=0.1, lmbda=0.1):
    
    l = math.ceil(math.log(4 / (eps * (1 - lmbda))) / math.log(1.0 / lmbda) / 2)
    r = int(math.ceil(40 * l * l * math.log(80 * l) / (eps * eps)))
    delta = np.zeros(len(t))
    
    #print(l,r)
    
    for i in range(l):
        
        Xis = 0; 
        Xit = {i:0 for i in t}; 
        Yis = {i:0 for i in t}; 
        Yit = {i:0 for i in t};
        
        for j in range(r):
            v = random_walk(G, s, i)
            
            if (v == s):
                Xis+=1
            
            if v in Xit:
                Xit[v]+=1
            
        for j in range(r):  
            
            for t_i in t:            
                v = random_walk(G, t_i, i);
                if (v == s):
                    Yis[t_i]+=1;
                
                if v in Yit:
                    Yit[v]+=1;
                    
        
        deltai = np.zeros(len(t))
        
        for it in range(len(t)):        
            deltai[it] = float(Xis) / G.degree[s] - float(Xit[t[it]]) / G.degree[t[it]] \
            - float(Yis[t[it]]) / G.degree[s] + float(Yit[t[it]]) / G.degree[t[it]]
            deltai[it] /= r;
            delta[it] += deltai[it];

    for i in range(len(delta)):
        delta[i]=max(0,delta[i])
    return delta

#local_effective_resistance_akp(G, 21, [0,20], eps=0.1, lmbda=0.1)    

In [72]:
def ERfast(G, window=50):
        
    Re = nx.adjacency_matrix(G, dtype=float)
    
    nodes = G.nodes()
    N=len(nodes)
    
    pbar = tqdm(total=N)
    pbar.set_description(f'Nodes')
    
    last_i=-1
    
    for it,u in enumerate(nodes):    
        neighbors=list(G.neighbors(u))

        re_weight=local_effective_resistance_akp(G, u, neighbors, eps=0.1, lmbda=0.1)
        
        for i,v in enumerate(neighbors):
            Re[u,v]=re_weight[i]
                
        
        if (it%window==0 and it>0) or it==(N-1):
            pbar.update(it-last_i)
            last_i=it
        
    pbar.close()
        
    return Re
    
#Re = ERfast(G, window=4)
#print(Re)

In [73]:
def LocalERfast(G, epsilon=0.5, computation='parallel', window=100):
        
    
    Adj = nx.adjacency_matrix(G)
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(Adj))
    
    print("ER computation: ",computation)
    
    AdjRe=ERfast(G,window)
    start_nodes, end_nodes, Re = sparse.find(sparse.tril(AdjRe))    
    
    N=Adj.shape[0]
    
    # Calculate the new weights.
    weights = np.maximum(0, weights)
    Pe = weights * Re
    Pe = Pe / np.sum(Pe)    
    Pe = np.squeeze(Pe)
    
    # Rudelson, 1996 Random Vectors in the Isotropic Position
    # (too hard to figure out actual C0)
    C0 = 1 / 30.
    # Rudelson and Vershynin, 2007, Thm. 3.1
    C = 4 * C0
    q = round(N * np.log(N) * 9 * C ** 2 / (epsilon ** 2))
    
#     print(len(start_nodes))
#     print(C0,C,q)

    #        results = stats.rv_discrete(values=(np.arange(np.shape(Pe)[0]), Pe)).rvs(size=int(q))
    results = np.random.choice(np.arange(np.shape(Pe)[0]), int(q), p=list(Pe))
    #spin_counts = stats.itemfreq(results).astype(int)
    spin_counts = np.unique(results, return_counts=True)
    
    #print(results, spin_counts)
    
    per_spin_weights = weights / (q * Pe)
    per_spin_weights[per_spin_weights == inf] = 0
    
    #print(per_spin_weights)

    counts = np.zeros(np.shape(weights)[0])
    #counts[spin_counts[:, 0]] = spin_counts[:, 1]    
    counts[spin_counts[0]] = spin_counts[1]
    
    #print(counts)
    
    new_weights = counts * per_spin_weights

    sparserW = sparse.csc_matrix((np.squeeze(new_weights), (start_nodes, end_nodes)),shape=(N, N))    
    sparserW = sparserW + sparserW.T ##making symmetric
    
    #print(N,q)

    return sparserW, np.count_nonzero(new_weights)
    

# sparserW, nz = LocalERfast(G,0.5, computation='parallel', window=NUM_PROCESSORS*100)
# print(nz) ##directed edges
# Gsparse = nx.from_scipy_sparse_matrix(sparserW)
# BGraph.draw_graph(Gsparse,pos)